In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("Set2")

In [ ]:
# Load the data
df = pd.read_csv("../data/ethiopia.csv")  # Adjust filename if different

# Add country column
df["Country"] = "Ethiopia"

# Check first few rows
df.head()

In [ ]:
print(f"Dataset shape: {df.shape}")
print("\nColumn info:")
df.info()

In [ ]:
# Replace NASA sentinel value -999 with NaN
df.replace(-999, np.nan, inplace=True)

print("Replaced -999 with NaN")
print(f"Missing values after replacement:\n{df.isna().sum()}")

In [ ]:
# Convert YEAR and DOY to datetime
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract month
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year

# Check result
df[["YEAR", "DOY", "Date", "Month", "Year"]].head()

In [ ]:
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Drop duplicates if any
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"Dropped {duplicates} duplicates")

In [ ]:
# Describe numeric columns
df.describe()

In [ ]:
# Calculate missing values
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage': missing_pct
})

print(missing_report[missing_report['Missing Count'] > 0])

In [ ]:
## Data Interpretation

### Summary Statistics Observations:
- Mean temperature in Ethiopia is approximately [fill from describe output] °C
- Maximum temperature reaches up to [fill from describe output] °C
- Average daily precipitation is [fill from describe output] mm/day

### Missing Values:
- Columns with >5% missing values: [list columns if any]
- This might affect analysis of [explain implications]

### Data Quality Assessment:
- No duplicate rows found
- -999 values successfully replaced with NaN
- Date parsing completed successfully

In [ ]:
# Columns to check for outliers
outlier_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']

# Calculate Z-scores
outlier_counts = {}
for col in outlier_cols:
    if col in df.columns:
        z_scores = np.abs(stats.zscore(df[col].dropna()))
        outliers = (z_scores > 3).sum()
        outlier_counts[col] = outliers
        print(f"{col}: {outliers} outliers")

# Flag outlier rows
outlier_rows = pd.Series(False, index=df.index)
for col in outlier_cols:
    if col in df.columns:
        z_scores = np.abs(stats.zscore(df[col]))
        outlier_rows |= (z_scores > 3)

print(f"\nTotal rows with at least one outlier: {outlier_rows.sum()}")

In [ ]:
## Outlier Handling Decision

Found [number] rows with outliers in columns: [list columns]

**Decision:** [Choose one]
- Option A: Keep outliers (if they represent real extreme weather events)
- Option B: Cap outliers at ±3 standard deviations
- Option C: Remove outlier rows (if less than 5% of data)

**Reasoning:** [Explain your choice]

For Ethiopia climate data, extreme values may represent actual heatwaves or heavy rainfall events, so I will [state your decision].

In [ ]:
# Forward fill for weather variables
weather_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']

for col in weather_cols:
    if col in df.columns:
        df[col] = df[col].fillna(method='ffill')

# Check remaining missing values
print(f"Remaining missing values:\n{df.isna().sum()}")

In [ ]:
# Save to data folder
df.to_csv("../data/ethiopia_clean.csv", index=False)
print("Saved cleaned data to data/ethiopia_clean.csv")
print(f"Final shape: {df.shape}")

In [ ]:
# Create monthly average temperature
monthly_temp = df.groupby(['Year', 'Month'])['T2M'].mean().reset_index()
monthly_temp['Date'] = pd.to_datetime(monthly_temp['Year'].astype(str) + '-' + monthly_temp['Month'].astype(str) + '-01')

# Plot
plt.figure(figsize=(14, 6))
plt.plot(monthly_temp['Date'], monthly_temp['T2M'], linewidth=1)

# Annotate warmest and coolest months
warmest = monthly_temp.loc[monthly_temp['T2M'].idxmax()]
coolest = monthly_temp.loc[monthly_temp['T2M'].idxmin()]

plt.scatter(warmest['Date'], warmest['T2M'], color='red', s=100, label=f'Warmest: {warmest["T2M"]:.1f}°C')
plt.scatter(coolest['Date'], coolest['T2M'], color='blue', s=100, label=f'Coolest: {coolest["T2M"]:.1f}°C')

plt.title('Ethiopia: Monthly Average Temperature (2015-2026)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Create monthly average temperature
monthly_temp = df.groupby(['Year', 'Month'])['T2M'].mean().reset_index()
monthly_temp['Date'] = pd.to_datetime(monthly_temp['Year'].astype(str) + '-' + monthly_temp['Month'].astype(str) + '-01')

# Plot
plt.figure(figsize=(14, 6))
plt.plot(monthly_temp['Date'], monthly_temp['T2M'], linewidth=1)

# Annotate warmest and coolest months
warmest = monthly_temp.loc[monthly_temp['T2M'].idxmax()]
coolest = monthly_temp.loc[monthly_temp['T2M'].idxmin()]

plt.scatter(warmest['Date'], warmest['T2M'], color='red', s=100, label=f'Warmest: {warmest["T2M"]:.1f}°C')
plt.scatter(coolest['Date'], coolest['T2M'], color='blue', s=100, label=f'Coolest: {coolest["T2M"]:.1f}°C')

plt.title('Ethiopia: Monthly Average Temperature (2015-2026)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Create monthly total precipitation
monthly_rain = df.groupby(['Year', 'Month'])['PRECTOTCORR'].sum().reset_index()
monthly_rain['Date'] = pd.to_datetime(monthly_rain['Year'].astype(str) + '-' + monthly_rain['Month'].astype(str) + '-01')

# Plot
plt.figure(figsize=(14, 6))
plt.bar(monthly_rain['Date'], monthly_rain['PRECTOTCORR'], width=25)

# Find peak rainy season
peak_month = monthly_rain.groupby('Month')['PRECTOTCORR'].mean().idxmax()

plt.title(f'Ethiopia: Monthly Total Precipitation (2015-2026)\nPeak Rainy Season: Month {peak_month}', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Precipitation (mm)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Peak rainy season month: {peak_month}")

In [ ]:
## Time Series Observations

### Temperature Trends:
- Ethiopia's warmest month reached [value]°C in [date]
- Coolest month was [value]°C in [date]
- [Observe any upward/downward trend]

### Precipitation Patterns:
- Peak rainy season occurs in month [number]
- Wettest period: [describe]
- [Note any changes in rainfall patterns over years]

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'T2M_RANGE', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX', 'PS', 'QV2M']
corr_matrix = df[numeric_cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Ethiopia: Correlation Matrix of Climate Variables')
plt.show()

# Find top 3 correlations
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1]  # Remove self-correlations
print("\nTop 3 Strongest Correlations:")
print(corr_pairs.head(3))

In [ ]:
# Scatter plot 1: Temperature vs Humidity
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(df['T2M'], df['RH2M'], alpha=0.3, s=1)
plt.xlabel('Temperature (°C)')
plt.ylabel('Relative Humidity (%)')
plt.title('T2M vs RH2M')

# Scatter plot 2: Temperature Range vs Wind Speed
plt.subplot(1, 2, 2)
plt.scatter(df['T2M_RANGE'], df['WS2M'], alpha=0.3, s=1)
plt.xlabel('Temperature Range (°C)')
plt.ylabel('Wind Speed (m/s)')
plt.title('T2M_RANGE vs WS2M')

plt.tight_layout()
plt.show()

In [ ]:
## Correlation Analysis Findings

**Three Strongest Correlations:**

1. [Variable A] vs [Variable B]: [correlation value]
   - Interpretation: [Explain what this means for Ethiopia's climate]

2. [Variable C] vs [Variable D]: [correlation value]
   - Interpretation: [Explain]

3. [Variable E] vs [Variable F]: [correlation value]
   - Interpretation: [Explain]

**Key Insight:** [Summarize what these correlations tell you about Ethiopia's climate]

In [ ]:
# Histogram of precipitation
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df['PRECTOTCORR'].dropna(), bins=50, edgecolor='black')
plt.xlabel('Daily Precipitation (mm/day)')
plt.ylabel('Frequency')
plt.title('Precipitation Distribution (Original)')

# Log scale if skewed
plt.subplot(1, 2, 2)
plt.hist(np.log1p(df['PRECTOTCORR'].dropna()), bins=50, edgecolor='black')
plt.xlabel('Log(Precipitation + 1)')
plt.ylabel('Frequency')
plt.title('Precipitation Distribution (Log Scale)')

plt.tight_layout()
plt.show()

# Skewness
skew = df['PRECTOTCORR'].skew()
print(f"Skewness of precipitation data: {skew:.2f}")

In [ ]:
# Bubble chart: Temperature vs Humidity, bubble size = Precipitation
# Sample every 30th row to avoid overcrowding
sample = df.dropna(subset=['T2M', 'RH2M', 'PRECTOTCORR']).iloc[::30]

plt.figure(figsize=(12, 8))
scatter = plt.scatter(sample['T2M'], sample['RH2M'], 
                      s=sample['PRECTOTCORR'] * 10,  # Scale bubble size
                      alpha=0.6, c=sample['PRECTOTCORR'], 
                      cmap='YlOrRd')

plt.colorbar(scatter, label='Precipitation (mm/day)')
plt.xlabel('Temperature (°C)')
plt.ylabel('Relative Humidity (%)')
plt.title('Ethiopia: Temperature vs Humidity (Bubble size = Precipitation)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
## Distribution Analysis Findings

### Precipitation Distribution:
- The data is [skewed/not skewed] with skewness value of [value]
- Most days have [low/high] precipitation
- Log scale reveals [describe pattern]

### Bubble Chart Insights:
- Higher precipitation occurs when temperature is [range] and humidity is [range]
- [Additional observations about Ethiopia's climate patterns]

In [ ]:
print("EDA for Ethiopia completed successfully!")
print(f"Cleaned data saved to: data/ethiopia_clean.csv")
print(f"Total records: {len(df)}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")